# Windows / VS Code environment and real-render acceptance test

**Setup only: no new development Part.** Follow `docs/WINDOWS_SETUP.md` first.
In VS Code choose **Select Kernel → Python Environments → quonitron-manim-render**
(or the registered QUONITRON kernel). The Python extension interpreter and the
notebook kernel are separate selections; inspect `sys.executable` below.

Run cells in order. Missing dependencies **raise an error**, not a successful skip.
Rendering uses the kernel's `sys.executable -m manim` in a subprocess. This avoids
assuming `%%manim` exists, isolates configuration, and reloads scene code on every run.
`IPython.display.Video` embeds MP4 where the frontend supports it; only you can
confirm actual Windows browser playback and readable subtitles.

In [ ]:
import sys, json, importlib.util
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "src/quonitron_manim").is_dir(), "Open the project folder, not an unrelated working directory"
def load_script(name):
    spec = importlib.util.spec_from_file_location(name, ROOT / "scripts" / (name + ".py"))
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module
diagnostics = load_script("verify_environment")
report = diagnostics.collect()
print("Kernel:", sys.executable)
print("Platform:", report["os"], report["architecture"])
for check in report["checks"]:
    print(("PASS" if check["ok"] else "FAIL" if check["required"] else "OPTIONAL MISSING"), check["name"], check["detail"])
print("Environment status:", report["status"])
if not report["ready"]:
    raise RuntimeError("BLOCKED: dependency verification failed. No render or playback is verified. See docs/WINDOWS_SETUP.md; fix the selected kernel before continuing.")

## Real render / decode / inline helper
Each invocation uses a fresh output directory (no stale movie acceptance), validates
process exit, requires a non-empty MP4, decodes it with FFmpeg and extracts PNGs.
All failures raise, including missing frames. Logs remain in the printed directory.
After editing a scene file, rerun its render cell; restarting the kernel is not
required for subprocess scene reloads. No global Manim configuration is changed.

In [ ]:
from datetime import datetime, timezone
from IPython.display import Video, Image, display
renderer = load_script("verify_renders")
def render_and_preview(key, quality="low"):
    if not report["ready"]:
        raise RuntimeError("BLOCKED: environment verification did not pass")
    run = ROOT / "media/notebook-environment" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    result = renderer.render_case(key, run, quality=quality)
    print(json.dumps(result, indent=2))
    if result["status"] != "RENDERED_AND_DECODED_NOT_VISUALLY_REVIEWED":
        log = Path(result["log"])
        if log.exists(): print(log.read_text(encoding="utf-8")[-8000:])
        raise RuntimeError("Real render/decode/frame verification FAILED; inspect logs above")
    video = Video(filename=result["movie"], embed=True)
    html = video._repr_html_()
    assert "video" in html and "base64" in html, "Embedded MP4 HTML was not generated"
    display(video)
    print("MP4 rendered and decoded; HTML generated. Browser playback still requires visual confirmation.")
    for frame in result["frames"]:
        print("Timestamp:", frame["time"])
        display(Image(filename=frame["path"]))
    return result

## 1. Minimal scene: Cairo + Pango + video encoding
`MinimalRenderCheck` in `examples/environment_smoke.py` draws a circle and text.
No LaTeX, audio, OpenGL, or new component is required.

In [ ]:
minimal_result = render_and_preview("minimal")

## 2. Existing QUONITRON theme, subtitle and layout safety
`ThemeSubtitleRenderCheck` uses the existing scene helpers, measured stack,
subtitle-safe region, and validator. Inspect the title, lesson, circle, and caption.
The caption must not cover the lesson.

In [ ]:
theme_result = render_and_preview("themed")

## 3. Overlap repair exercise: long text, labeled diagram and transition
This is the same exercise as the existing overlap repair lab. Inspect the PNGs
and video, especially the sequential paragraph-to-diagram transition.
Its final deliberately conflicting label pair prints an ERROR diagnostic to the
render log; that intentional test is distinct from a rendering-process failure.

In [ ]:
overlap_result = render_and_preview("overlap")

## 4. Existing Parts 1–4
These render full demos (roughly several minutes total, depending on hardware).
Font-dependent layout or application errors are not hidden. Read tracebacks in
the printed `render.log`; do not try to fix application errors by reinstalling
packages unless the error actually identifies a dependency failure.

In [ ]:
foundation_result = render_and_preview("foundation")

In [ ]:
motion_result = render_and_preview("motion")

In [ ]:
transition_result = render_and_preview("transitions")

In [ ]:
typography_result = render_and_preview("typography")

## Optional intended-quality check (1920×1080, 30 fps)
After low-quality checks pass, run `render_and_preview("themed", quality="production")`.
This is an explicit opt-in because full production renders can take much longer.

## Troubleshooting and acceptance
- **Wrong interpreter/import:** inspect `sys.executable`; select the dedicated kernel, restart it, rerun diagnostics. Install only this project editable with `--no-deps` in the Conda environment.
- **DLL/Cairo/Pango:** launch VS Code from the activated Miniforge environment (`code .`). Do not copy arbitrary DLLs or pip-upgrade Conda's compiled packages.
- **FFmpeg:** `conda run -n quonitron-manim-render ffmpeg -version`. Needed by this repository's checks, not by Manim 0.19's PyAV encoder.
- **LaTeX:** optional. EquationText has a plain-text fallback. Actual MathTex requires LaTeX and dvisvgm; see the guide.
- **Video does not play:** open the printed MP4 externally. Check VS Code notebook trust and output policy. Successful HTML generation alone does not verify playback.
- **Re-run changes:** edit/save the `.py` scene, rerun its render cell. Old artifacts are retained in ignored `media/`; delete only your generated output when desired.
- **Layout ERROR:** scene logs identify elements; inspect frames before changing layout. This setup task does not certify previous overlap repairs.

Record your local result: kernel path, rendered scene, movie path, and whether you actually saw playback, caption readability, and transitions. **Do not mark acceptance complete just because tests skipped or diagnostics passed.**